[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/09-system-design/07_apis.ipynb)

*System Design · Notebook 7 of 10*

# APIs & Rate Limiting

Designing and protecting APIs: fixed-window, sliding-window, token-bucket and leaky-bucket limiters, per-user quotas, offset vs cursor pagination, idempotency keys, status codes, safe and idempotent methods, versioning, ETags, batching, deadlines, circuit breakers, gateways, webhook signatures and push vs poll.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A rate limiter is a ticket barrier: tokens refill at a steady rate, and when they run out you wait your turn.

**What interviewers really test here:** API design, status codes, pagination, idempotency keys, token bucket and sliding windows, circuit breakers and gateways.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Designing an API *(Basics)*
- [ ] 2. Pagination, idempotency and caching *(Core)*
- [ ] 3. Rate limiting *(Core)*
- [ ] 4. Resilience at the edge *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### REST vs gRPC vs GraphQL

|   | REST | gRPC | GraphQL |
| --- | --- | --- | --- |
| Format | JSON over HTTP | Protobuf over HTTP/2 | JSON, one endpoint |
| Best for | public APIs | internal, low latency, streaming | many client shapes, mobile |
| Caching | HTTP caching works | custom | harder |

---
# Part 2: Lessons, step by step

## Lesson 1: Designing an API

*Level: Basics*

In an interview, write the API right after requirements — it pins down what the system does.



```python
POST   /v1/orders               body {items, address}  Idempotency-Key: 7f3c…   → 201 {order_id}
GET    /v1/orders/{id}                                                         → 200 / 404
GET    /v1/users/{id}/orders?limit=20&cursor=eyJpZCI6MTAyfQ                → 200 {items, next_cursor}
PATCH  /v1/orders/{id}          body {status}                                  → 200 / 409
```




- Nouns for resources, HTTP verbs for actions, plural paths, a version in the path or a header.
- Right status codes: 2xx success, 4xx the client's fault, 5xx yours; 429 with `Retry-After` when throttled.
- REST for public CRUD APIs, gRPC for fast internal service-to-service calls, GraphQL when clients need flexible shapes, WebSockets/SSE for server push.

**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q4](#q4), [Q5](#q5), [Q7](#q7). Try one now, or carry on and do them in Part 3.

## Lesson 2: Pagination, idempotency and caching

*Level: Core*

- **Offset** pagination (`LIMIT 20 OFFSET 4000`) is simple but slow deep down and skips or repeats rows as data changes. **Cursor/keyset** pagination (`WHERE (created_at, id) < cursor`) is stable and uses the index.
- **Idempotency keys** make POST safe to retry: store the key with the response and replay it.
- **ETags** let clients revalidate: send `If-None-Match`, get 304 with no body if nothing changed.
- Reduce round trips: batch endpoints, or parallel calls with a concurrency limit.

**✏️ Your turn:** you now have what you need for [Q13](#q13), [Q14](#q14), [Q15](#q15), [Q16](#q16), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 3: Rate limiting

*Level: Core*

Rate limits protect your service from abuse and from one client starving the rest. Limit per user, API key or IP, usually at the gateway, with counters in Redis so all gateway nodes share them.



| Algorithm | Pros | Cons |
| --- | --- | --- |
| Fixed window | one counter, simple | 2× burst at window edges |
| Sliding log | exact | stores every timestamp |
| Sliding counter | two counters, smooth | approximate |
| Token bucket | allows controlled bursts | two values per client |
| Leaky bucket | perfectly smooth output | bursts are dropped or queued |

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def bucket_demo():
    tokens, last, cap, rate = 3, 0, 3, 1
    for t in [0, 0, 0, 0, 0.5, 1.5, 2]:
        tokens = min(cap, tokens + (t - last) * rate); last = t
        ok = tokens >= 1
        tokens -= ok
        print(f"t={t:<4} {'allowed' if ok else 'rejected (429)'}  tokens left={tokens:.1f}")


bucket_demo()

t=0    allowed  tokens left=2.0
t=0    allowed  tokens left=1.0
t=0    allowed  tokens left=0.0
t=0    rejected (429)  tokens left=0.0
t=0.5  rejected (429)  tokens left=0.5
t=1.5  allowed  tokens left=0.5
t=2    allowed  tokens left=0.0


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q8](#q8), [Q9](#q9), [Q10](#q10), [Q11](#q11), [Q12](#q12), [Q2](#q2). Try one now, or carry on and do them in Part 3.

## Lesson 4: Resilience at the edge

*Level: Advanced*

- **Deadlines**: give every request a total time budget and pass it downstream; retries must fit inside it.
- **Circuit breakers** stop calling a failing dependency for a while, so you fail fast and give it room to recover.
- An **API gateway** handles cross-cutting work: authentication, routing, rate limits, TLS, request logging.
- **Webhooks** push events to customers; sign them with HMAC so receivers can verify the sender, and retry with backoff.

**✏️ Your turn:** you now have what you need for [Q17](#q17), [Q20](#q20), [Q18](#q18), [Q19](#q19). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [3]:
from collections import Counter, defaultdict, deque

import bisect

import hashlib


<a id="q1"></a>
### Q1. Fixed-Window Rate Limiter

🟢 Warm-up · *Count per window*

Allow at most `limit` requests per window of `window` seconds (windows start at multiples of window). Return whether each request time is allowed.

**Examples**

- `times = [0, 1, 2, 3, 9.9, 10, 10.5, 11, 12, 25], limit = 3, window = 10` → `[True, True, True, False, False, True, True, True, False, True]`

In [ ]:
def fixed_window(times, limit, window):
    """
    Allow at most limit requests per window of window seconds (windows start at
    multiples of window). Return whether each request time is allowed.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fixed_window, [
    (([0, 1, 2, 3, 9.9, 10, 10.5, 11, 12, 25], 3, 10), [True, True, True, False, False, True, True, True, False, True]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count earlier allowed requests in the same window.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A counter keyed by window number.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count earlier allowed requests in the same window.

```python
def fixed_window(times, limit, window):
    allowed = []
    for i, t in enumerate(times):
        w = int(t // window)
        used = sum(1 for j in range(i) if allowed[j] and int(times[j] // window) == w)
        allowed.append(used < limit)
    return allowed
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A counter keyed by window number.

```python
def fixed_window(times, limit, window):
    counts, out = Counter(), []
    for t in times:
        w = int(t // window)
        out.append(counts[w] < limit)
        counts[w] += out[-1]
    return out
```

**Cost:** O(n) time, O(w) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count earlier allowed requests in the same window. That costs O(n²) time and O(n) space."

"I can do better: A counter keyed by window number. That's O(n) time and O(w) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/fixed-window-rate-limiter)

<a id="q2"></a>
### Q2. Per-User Quotas

🟢 Warm-up · *One noisy client shouldn't hurt others*

Requests are `(user, second)`. Each user may make `limit` requests per fixed minute (60 s). Return the rejected count per user (only users with rejections, sorted).

**Examples**

- `requests = [('u1', 1), ('u1', 2), ('u2', 3), ('u1', 4), ('u1', 30), ('u2', 59), ('u1', 61), ('u1', 62), ('u1', 63), ('u1', 64)], limit = 3` → `[('u1', 2)]`

In [ ]:
def user_quota(requests, limit):
    """
    Requests are (user, second). Each user may make limit requests per fixed
    minute (60 s). Return the rejected count per user (only users with
    rejections, sorted).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(user_quota, [
    (([('u1', 1), ('u1', 2), ('u2', 3), ('u1', 4), ('u1', 30), ('u2', 59), ('u1', 61), ('u1', 62), ('u1', 63), ('u1', 64)], 3), [('u1', 2)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count prior requests per user per minute.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A counter keyed by (user, minute).

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count prior requests per user per minute.

```python
def user_quota(requests, limit):
    seen, rejected = [], {}
    for u, t in requests:
        used = sum(1 for su, st in seen if su == u and st // 60 == t // 60)
        if used < limit:
            seen.append((u, t))
        else:
            rejected[u] = rejected.get(u, 0) + 1
    return sorted(rejected.items())
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A counter keyed by (user, minute).

```python
def user_quota(requests, limit):
    used, rejected = Counter(), Counter()
    for u, t in requests:
        if used[u, t // 60] < limit:
            used[u, t // 60] += 1
        else:
            rejected[u] += 1
    return sorted(rejected.items())
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count prior requests per user per minute. That costs O(n²) time and O(n) space."

"I can do better: A counter keyed by (user, minute). That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/per-user-quotas)

<a id="q3"></a>
### Q3. Pick the Status Code

🟢 Warm-up · *Speak HTTP*

Return the HTTP status code for each situation using `CODES`; anything unknown is 500.

**Examples**

- `situations = ['created', 'not logged in', 'not allowed', 'too many requests', 'version conflict', 'accepted for later', 'dependency down', 'who knows']` → `[201, 401, 403, 429, 409, 202, 503, 500]`

In [ ]:
CODES = {"created": 201, "accepted for later": 202, "deleted": 204, "bad input": 400, "not logged in": 401,
         "not allowed": 403, "not found": 404, "version conflict": 409, "too many requests": 429,
         "server bug": 500, "dependency down": 503}

def status_codes(situations):
    """
    Return the HTTP status code for each situation using CODES; anything unknown
    is 500.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(status_codes, [
    ((['created', 'not logged in', 'not allowed', 'too many requests', 'version conflict', 'accepted for later', 'dependency down', 'who knows'],), [201, 401, 403, 429, 409, 202, 503, 500]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if/elif chain.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`dict.get` with a default.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** An if/elif chain.

```python
def status_codes(situations):
    out = []
    for s in situations:
        code = 500
        for k, v in CODES.items():
            if k == s:
                code = v
        out.append(code)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `dict.get` with a default.

```python
def status_codes(situations):
    return [CODES.get(s, 500) for s in situations]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if/elif chain. That costs O(n) time and O(n) space."

"I can do better: `dict.get` with a default. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/pick-the-status-code)

<a id="q4"></a>
### Q4. Safe and Idempotent Methods

🟢 Warm-up · *Which calls can be retried blindly?*

Return `(method, safe, idempotent)`: GET, HEAD and OPTIONS are safe (no changes) and idempotent; PUT and DELETE are idempotent but not safe; POST and PATCH are neither.

**Examples**

- `methods = ['GET', 'POST', 'PUT', 'PATCH', 'DELETE', 'HEAD']` → `[('GET', True, True), ('POST', False, False), ('PUT', False, True), ('PATCH', False, False), ('DELETE', False, True), ('HEAD', True, True)]`

In [ ]:
def method_props(methods):
    """
    Return (method, safe, idempotent): GET, HEAD and OPTIONS are safe (no
    changes) and idempotent; PUT and DELETE are idempotent but not safe; POST
    and PATCH are neither.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(method_props, [
    ((['GET', 'POST', 'PUT', 'PATCH', 'DELETE', 'HEAD'],), [('GET', True, True), ('POST', False, False), ('PUT', False, True), ('PATCH', False, False), ('DELETE', False, True), ('HEAD', True, True)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

If/elif per method.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Set membership.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** If/elif per method.

```python
def method_props(methods):
    out = []
    for m in methods:
        if m in ("GET", "HEAD", "OPTIONS"):
            out.append((m, True, True))
        elif m in ("PUT", "DELETE"):
            out.append((m, False, True))
        else:
            out.append((m, False, False))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Set membership.

```python
def method_props(methods):
    safe = {"GET", "HEAD", "OPTIONS"}
    return [(m, m in safe, m in safe | {"PUT", "DELETE"}) for m in methods]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: If/elif per method. That costs O(n) time and O(n) space."

"I can do better: Set membership. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/safe-and-idempotent-methods)

<a id="q5"></a>
### Q5. Route by API Version

🟢 Warm-up · *Serve the newest compatible version*

The server implements the sorted `versions`. A client asking for version v gets the highest implemented version ≤ v; below the oldest supported version it gets '410 Gone'.

**Examples**

- `versions = [2, 3, 5, 8], requested = [1, 2, 4, 5, 7, 9]` → `['410 Gone', 2, 3, 5, 5, 8]`

In [ ]:
def api_version(versions, requested):
    """
    The server implements the sorted versions. A client asking for version v
    gets the highest implemented version ≤ v; below the oldest supported version
    it gets '410 Gone'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(api_version, [
    (([2, 3, 5, 8], [1, 2, 4, 5, 7, 9]), ['410 Gone', 2, 3, 5, 5, 8]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan the versions.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`bisect_right`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Scan the versions.

```python
def api_version(versions, requested):
    out = []
    for r in requested:
        best = None
        for v in versions:
            if v <= r:
                best = v
        out.append(best if best is not None else "410 Gone")
    return out
```

**Cost:** O(r · v) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `bisect_right`.

```python
def api_version(versions, requested):
    def pick(r):
        i = bisect.bisect_right(versions, r)
        return versions[i - 1] if i else "410 Gone"
    return [pick(r) for r in requested]
```

**Cost:** O(r log v) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan the versions. That costs O(r · v) time and O(1) space."

"I can do better: `bisect_right`. That's O(r log v) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/route-by-api-version)

<a id="q6"></a>
### Q6. Chatty vs Batched vs Parallel Calls

🟢 Warm-up · *Round trips add up*

Fetching `n` items costs one round trip of `rtt` ms per call plus `item_ms` per item served. Return the total ms for one call per item in sequence, batches of `batch` in sequence, and one call per item with `concurrency` in flight.

**Examples**

- `n = 100, rtt = 40, item_ms = 1, batch = 25, concurrency = 10` → `(4100, 260, 410)`

In [ ]:
def call_patterns(n, rtt, item_ms, batch, concurrency):
    """
    Fetching n items costs one round trip of rtt ms per call plus item_ms per
    item served. Return the total ms for one call per item in sequence, batches
    of batch in sequence, and one call per item with concurrency in flight.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(call_patterns, [
    ((100, 40, 1, 25, 10), (4100, 260, 410)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loops adding up time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Closed-form with `math.ceil`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loops adding up time.

```python
def call_patterns(n, rtt, item_ms, batch, concurrency):
    seq = 0
    for _ in range(n):
        seq += rtt + item_ms
    batched, left = 0, n
    while left > 0:
        k = min(batch, left)
        batched += rtt + k * item_ms
        left -= k
    waves = 0
    left = n
    while left > 0:
        waves += 1
        left -= concurrency
    return (seq, batched, waves * (rtt + item_ms))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Closed-form with `math.ceil`.

```python
def call_patterns(n, rtt, item_ms, batch, concurrency):
    return (n * (rtt + item_ms), math.ceil(n / batch) * rtt + n * item_ms, math.ceil(n / concurrency) * (rtt + item_ms))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loops adding up time. That costs O(n) time and O(1) space."

"I can do better: Closed-form with `math.ceil`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/chatty-vs-batched-vs-parallel-calls)

<a id="q7"></a>
### Q7. Polling vs Push

🟢 Warm-up · *Wasted requests*

`clients` each poll every `poll_s` seconds, but each client only gets `updates_per_min` real updates per minute. Return `(poll requests/s, pushed messages/s over WebSockets/SSE, % of polls that find nothing)`, rounded to 1.

**Examples**

- `clients = 100000, poll_s = 5, updates_per_min = 1` → `(20000.0, 1666.7, 91.7)`
- `clients = 2000, poll_s = 30, updates_per_min = 4` → `(66.7, 133.3, 0.0)`

In [ ]:
def poll_vs_push(clients, poll_s, updates_per_min):
    """
    clients each poll every poll_s seconds, but each client only gets
    updates_per_min real updates per minute. Return (poll requests/s, pushed
    messages/s over WebSockets/SSE, % of polls that find nothing), rounded to 1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(poll_vs_push, [
    ((100000, 5, 1), (20000.0, 1666.7, 91.7)),
    ((2000, 30, 4), (66.7, 133.3, 0.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step by step arithmetic.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, compact.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Step by step arithmetic.

```python
def poll_vs_push(clients, poll_s, updates_per_min):
    polls = clients / poll_s
    pushes = clients * updates_per_min / 60
    useful = pushes
    if useful > polls:
        useful = polls
    wasted = (polls - useful) / polls * 100
    return (round(polls, 1), round(pushes, 1), round(wasted, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, compact.

```python
def poll_vs_push(clients, poll_s, updates_per_min):
    polls, pushes = clients / poll_s, clients * updates_per_min / 60
    return (round(polls, 1), round(pushes, 1), round((1 - min(pushes, polls) / polls) * 100, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step by step arithmetic. That costs O(1) time and O(1) space."

"I can do better: Same, compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/polling-vs-push)

<a id="q8"></a>
### Q8. The Window-Boundary Burst

🟡 Interview standard · *Fixed windows leak*

With a fixed-window limiter, requests at the end of one window and the start of the next all pass. Return the largest number of allowed requests inside any span of `window` seconds (span [t, t + window)).

**Examples**

- `times = [5, 9, 9.5, 9.8, 10, 10.1, 10.2, 14, 25], limit = 3, window = 10` → `6`

In [ ]:
def boundary_burst(times, limit, window):
    """
    With a fixed-window limiter, requests at the end of one window and the start
    of the next all pass. Return the largest number of allowed requests inside
    any span of window seconds (span [t, t + window)).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(boundary_burst, [
    (([5, 9, 9.5, 9.8, 10, 10.1, 10.2, 14, 25], 3, 10), 6),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Apply the limiter, then check every start point.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Apply the limiter, then a two-pointer sweep.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Apply the limiter, then check every start point.

```python
def boundary_burst(times, limit, window):
    allowed, counts = [], {}
    for t in times:
        w = int(t // window)
        if counts.get(w, 0) < limit:
            counts[w] = counts.get(w, 0) + 1
            allowed.append(t)
    best = 0
    for start in allowed:
        best = max(best, sum(1 for t in allowed if start <= t < start + window))
    return best
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Apply the limiter, then a two-pointer sweep.

```python
def boundary_burst(times, limit, window):
    counts, allowed = Counter(), []
    for t in times:
        w = int(t // window)
        if counts[w] < limit:
            counts[w] += 1
            allowed.append(t)
    best, lo = 0, 0
    for hi, t in enumerate(allowed):
        while allowed[lo] <= t - window:
            lo += 1
        best = max(best, hi - lo + 1)
    return best
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Apply the limiter, then check every start point. That costs O(n²) time and O(n) space."

"I can do better: Apply the limiter, then a two-pointer sweep. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/the-window-boundary-burst)

<a id="q9"></a>
### Q9. Sliding-Window Log

🟡 Interview standard · *Exact, but stores every timestamp*

Allow a request if fewer than `limit` allowed requests happened in the last `window` seconds (t − window, t]. Return the decisions.

**Examples**

- `times = [0, 1, 2, 3, 9.9, 10, 10.5, 11, 12, 25], limit = 3, window = 10` → `[True, True, True, False, False, True, False, True, True, True]`

In [ ]:
def sliding_log(times, limit, window):
    """
    Allow a request if fewer than limit allowed requests happened in the last
    window seconds (t − window, t]. Return the decisions.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sliding_log, [
    (([0, 1, 2, 3, 9.9, 10, 10.5, 11, 12, 25], 3, 10), [True, True, True, False, False, True, False, True, True, True]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan all previous allowed times.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A deque, popping expired timestamps.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Scan all previous allowed times.

```python
def sliding_log(times, limit, window):
    log, out = [], []
    for t in times:
        recent = [x for x in log if x > t - window]
        ok = len(recent) < limit
        if ok:
            log.append(t)
        out.append(ok)
    return out
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A deque, popping expired timestamps.

```python
def sliding_log(times, limit, window):
    log, out = deque(), []
    for t in times:
        while log and log[0] <= t - window:
            log.popleft()
        out.append(len(log) < limit)
        if out[-1]:
            log.append(t)
    return out
```

**Cost:** O(n) time, O(limit) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan all previous allowed times. That costs O(n²) time and O(n) space."

"I can do better: A deque, popping expired timestamps. That's O(n) time and O(limit) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/sliding-window-log)

<a id="q10"></a>
### Q10. Sliding-Window Counter

🟡 Interview standard · *Approximate and cheap*

Keep only two counters: the previous and current fixed window. Estimate = previous × (1 − elapsed fraction of the current window) + current. Allow if the estimate &lt; `limit`. Return the decisions.

**Examples**

- `times = [1, 2, 3, 4, 5, 11, 12, 13, 15, 18, 19], limit = 4, window = 10` → `[True, True, True, True, False, True, False, True, False, True, True]`

In [ ]:
def sliding_counter(times, limit, window):
    """
    Keep only two counters: the previous and current fixed window. Estimate =
    previous × (1 − elapsed fraction of the current window) + current. Allow if
    the estimate limit. Return the decisions.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sliding_counter, [
    (([1, 2, 3, 4, 5, 11, 12, 13, 15, 18, 19], 4, 10), [True, True, True, True, False, True, False, True, False, True, True]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recount the previous window each time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Roll two counters forward.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Recount the previous window each time.

```python
def sliding_counter(times, limit, window):
    allowed_times, out = [], []
    for t in times:
        w = int(t // window)
        prev = sum(1 for x in allowed_times if int(x // window) == w - 1)
        cur = sum(1 for x in allowed_times if int(x // window) == w)
        est = prev * (1 - (t - w * window) / window) + cur
        ok = est < limit
        if ok:
            allowed_times.append(t)
        out.append(ok)
    return out
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Roll two counters forward.

```python
def sliding_counter(times, limit, window):
    cur_w, prev, cur, out = None, 0, 0, []
    for t in times:
        w = int(t // window)
        if w != cur_w:
            prev = cur if cur_w == w - 1 else 0
            cur_w, cur = w, 0
        ok = prev * (1 - (t - w * window) / window) + cur < limit
        cur += ok
        out.append(ok)
    return out
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recount the previous window each time. That costs O(n²) time and O(n) space."

"I can do better: Roll two counters forward. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/sliding-window-counter)

<a id="q11"></a>
### Q11. Token Bucket

🟡 Interview standard · *Bursts up to a cap*

A bucket holds up to `capacity` tokens (starting full) and refills at `rate` tokens/s. Each request spends one token or is rejected. Return the decisions.

**Examples**

- `times = [0, 0, 0, 0, 0, 0.5, 1, 1.2, 3, 3, 3, 3], capacity = 4, rate = 1` → `[True, True, True, True, False, False, True, False, True, True, False, False]`

In [ ]:
def token_bucket(times, capacity, rate):
    """
    A bucket holds up to capacity tokens (starting full) and refills at rate
    tokens/s. Each request spends one token or is rejected. Return the
    decisions.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(token_bucket, [
    (([0, 0, 0, 0, 0, 0.5, 1, 1.2, 3, 3, 3, 3], 4, 1), [True, True, True, True, False, False, True, False, True, True, False, False]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Refill tick by tick in 0.1 s steps.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Refill lazily by elapsed time.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Refill tick by tick in 0.1 s steps.

```python
def token_bucket(times, capacity, rate):
    tokens, clock, out = capacity, 0, []
    for t in times:
        steps = round((t - clock) * 10)
        for _ in range(steps):
            tokens = min(capacity, tokens + rate / 10)
        clock = t
        if tokens >= 1 - 1e-9:
            tokens -= 1
            out.append(True)
        else:
            out.append(False)
    return out
```

**Cost:** O(T) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Refill lazily by elapsed time.

```python
def token_bucket(times, capacity, rate):
    tokens, last, out = capacity, 0, []
    for t in times:
        tokens = min(capacity, tokens + (t - last) * rate)
        last = t
        ok = tokens >= 1 - 1e-9
        tokens -= ok
        out.append(ok)
    return out
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Refill tick by tick in 0.1 s steps. That costs O(T) time and O(1) space."

"I can do better: Refill lazily by elapsed time. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/token-bucket)

<a id="q12"></a>
### Q12. Leaky Bucket

🟡 Interview standard · *A smooth output rate*

Requests pour into a bucket of `capacity` that leaks at `rate` per second; a request that would overflow is dropped. Return the decisions — output is smoothed to the leak rate.

**Examples**

- `times = [0, 0, 0, 0, 0, 1, 1, 1, 4, 4], capacity = 3, rate = 1` → `[True, True, True, False, False, True, False, False, True, True]`

In [ ]:
def leaky_bucket(times, capacity, rate):
    """
    Requests pour into a bucket of capacity that leaks at rate per second; a
    request that would overflow is dropped. Return the decisions — output is
    smoothed to the leak rate.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(leaky_bucket, [
    (([0, 0, 0, 0, 0, 1, 1, 1, 4, 4], 3, 1), [True, True, True, False, False, True, False, False, True, True]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Leak in 0.1 s steps.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Leak lazily by elapsed time.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Leak in 0.1 s steps.

```python
def leaky_bucket(times, capacity, rate):
    level, clock, out = 0.0, 0, []
    for t in times:
        for _ in range(round((t - clock) * 10)):
            level = max(0.0, level - rate / 10)
        clock = t
        if level + 1 <= capacity + 1e-9:
            level += 1
            out.append(True)
        else:
            out.append(False)
    return out
```

**Cost:** O(T) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Leak lazily by elapsed time.

```python
def leaky_bucket(times, capacity, rate):
    level, last, out = 0.0, 0, []
    for t in times:
        level, last = max(0.0, level - (t - last) * rate), t
        ok = level + 1 <= capacity + 1e-9
        level += ok
        out.append(ok)
    return out
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Leak in 0.1 s steps. That costs O(T) time and O(1) space."

"I can do better: Leak lazily by elapsed time. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/leaky-bucket)

<a id="q13"></a>
### Q13. Offset vs Cursor Pagination

🟡 Interview standard · *Pages that don't skip rows*

Paging through sorted ids with page size `size`; after page 1 is read, the ids in `deleted` are removed. Return the ids seen on the first `pages` pages with OFFSET paging and with cursor paging (id &gt; last seen). Offset paging skips rows when earlier rows disappear.

**Examples**

- `ids = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], size = 4, deleted = [2, 3], pages = 3` → `([1, 2, 3, 4, 7, 8, 9, 10, 11, 12], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12])`

In [ ]:
def pagination(ids, size, deleted, pages):
    """
    Paging through sorted ids with page size size; after page 1 is read, the ids
    in deleted are removed. Return the ids seen on the first pages pages with
    OFFSET paging and with cursor paging (id > last seen). Offset paging skips
    rows when earlier rows disappear.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pagination, [
    (([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], 4, [2, 3], 3), ([1, 2, 3, 4, 7, 8, 9, 10, 11, 12], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Slice lists for both methods.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Slicing for offset, `bisect` for the cursor.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Slice lists for both methods.

```python
def pagination(ids, size, deleted, pages):
    offset_seen, cursor_seen = [], []
    data = list(ids)
    for p in range(pages):
        if p == 1:
            data = [i for i in data if i not in deleted]
        offset_seen += data[p * size:(p + 1) * size]
    data, last = list(ids), 0
    for p in range(pages):
        if p == 1:
            data = [i for i in data if i not in deleted]
        page = [i for i in data if i > last][:size]
        cursor_seen += page
        if page:
            last = page[-1]
    return (offset_seen, cursor_seen)
```

**Cost:** O(p · n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Slicing for offset, `bisect` for the cursor.

```python
def pagination(ids, size, deleted, pages):
    after = [i for i in ids if i not in set(deleted)]
    views = [ids] + [after] * (pages - 1)
    offset = [x for p, d in enumerate(views) for x in d[p * size:(p + 1) * size]]
    cursor, last = [], 0
    for d in views:
        k = bisect.bisect_right(d, last)
        page = d[k:k + size]
        cursor += page
        last = page[-1] if page else last
    return (offset, cursor)
```

**Cost:** O(p · log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Slice lists for both methods. That costs O(p · n) time and O(n) space."

"I can do better: Slicing for offset, `bisect` for the cursor. That's O(p · log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/offset-vs-cursor-pagination)

<a id="q14"></a>
### Q14. Keyset Pagination With Ties

🟡 Interview standard · *Sort by (created\_at, id)*

Feeds sort by time, but many items share a timestamp. Use the cursor `(created_at, id)` of the last item and fetch items strictly after it. Items are `(created_at, id)`; return all pages of size `size`.

**Examples**

- `items = [(100, 7), (101, 2), (100, 3), (102, 9), (101, 1), (100, 5), (103, 4)], size = 3` → `[[(100, 3), (100, 5), (100, 7)], [(101, 1), (101, 2), (102, 9)], [(103, 4)]]`

In [ ]:
def keyset(items, size):
    """
    Feeds sort by time, but many items share a timestamp. Use the cursor
    (created_at, id) of the last item and fetch items strictly after it. Items
    are (created_at, id); return all pages of size size.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(keyset, [
    (([(100, 7), (101, 2), (100, 3), (102, 9), (101, 1), (100, 5), (103, 4)], 3), [[(100, 3), (100, 5), (100, 7)], [(101, 1), (101, 2), (102, 9)], [(103, 4)]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort, then filter by tuple comparison for each page.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort once and bisect on the cursor tuple.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Sort, then filter by tuple comparison for each page.

```python
def keyset(items, size):
    items = sorted(items)
    pages, cursor = [], None
    while True:
        page = [x for x in items if cursor is None or x > cursor][:size]
        if not page:
            return pages
        pages.append(page)
        cursor = page[-1]
```

**Cost:** O(p · n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Sort once and bisect on the cursor tuple.

```python
def keyset(items, size):
    items = sorted(items)
    pages, i = [], 0
    while i < len(items):
        pages.append(items[i:i + size])
        i = bisect.bisect_right(items, pages[-1][-1])
    return pages
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort, then filter by tuple comparison for each page. That costs O(p · n) time and O(n) space."

"I can do better: Sort once and bisect on the cursor tuple. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/keyset-pagination-with-ties)

<a id="q15"></a>
### Q15. Idempotency Keys

🟡 Interview standard · *Safe POST retries*

Requests are `(idempotency_key, body)`. The first use creates order #1, #2, …; a repeat with the same body returns the saved response; a repeat with a different body returns '422 key reused'. Return each response.

**Examples**

- `requests = [('k1', '2 shoes'), ('k2', '1 bag'), ('k1', '2 shoes'), ('k1', '3 shoes'), ('k3', '1 hat'), ('k2', '1 bag')]` → `['201 order #1', '201 order #2', '201 order #1', '422 key reused', '201 order #3', '201 order #2']`

In [ ]:
def idempotency_keys(requests):
    """
    Requests are (idempotency_key, body). The first use creates order #1, #2, …;
    a repeat with the same body returns the saved response; a repeat with a
    different body returns '422 key reused'. Return each response.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(idempotency_keys, [
    (([('k1', '2 shoes'), ('k2', '1 bag'), ('k1', '2 shoes'), ('k1', '3 shoes'), ('k3', '1 hat'), ('k2', '1 bag')],), ['201 order #1', '201 order #2', '201 order #1', '422 key reused', '201 order #3', '201 order #2']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Search a list of saved (key, body, response).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dict key → (body, response).

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Search a list of saved (key, body, response).

```python
def idempotency_keys(requests):
    saved, out, n = [], [], 0
    for key, body in requests:
        found = None
        for k, b, r in saved:
            if k == key:
                found = (b, r)
        if found is None:
            n += 1
            resp = f"201 order #{n}"
            saved.append((key, body, resp))
            out.append(resp)
        elif found[0] == body:
            out.append(found[1])
        else:
            out.append("422 key reused")
    return out
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dict key → (body, response).

```python
def idempotency_keys(requests):
    saved, out = {}, []
    for key, body in requests:
        if key not in saved:
            saved[key] = (body, f"201 order #{len(saved) + 1}")
        b, resp = saved[key]
        out.append(resp if b == body else "422 key reused")
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Search a list of saved (key, body, response). That costs O(n²) time and O(n) space."

"I can do better: A dict key → (body, response). That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/idempotency-keys)

<a id="q16"></a>
### Q16. Conditional GET With ETags

🟡 Interview standard · *304 Not Modified*

The ETag of a resource is the first 8 hex characters of the MD5 of its content. For each request `(path, If-None-Match etag or None)`, return 304 if the tag matches, else `(200, etag)`.

**Examples**

- `resources = {'/a': 'hello', '/b': 'world'}, requests = [('/a', None), ('/a', '5d41402a'), ('/b', '5d41402a'), ('/b', '7d793037')]` → `[(200, '5d41402a'), 304, (200, '7d793037'), 304]`

In [ ]:
def etags(resources, requests):
    """
    The ETag of a resource is the first 8 hex characters of the MD5 of its
    content. For each request (path, If-None-Match etag or None), return 304 if
    the tag matches, else (200, etag).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(etags, [
    (({'/a': 'hello', '/b': 'world'}, [('/a', None), ('/a', '5d41402a'), ('/b', '5d41402a'), ('/b', '7d793037')]), [(200, '5d41402a'), 304, (200, '7d793037'), 304]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Hash the content on every request.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compute each ETag once.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Hash the content on every request.

```python
def etags(resources, requests):
    out = []
    for path, tag in requests:
        etag = hashlib.md5(resources[path].encode()).hexdigest()[:8]
        if tag == etag:
            out.append(304)
        else:
            out.append((200, etag))
    return out
```

**Cost:** O(r · L) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compute each ETag once.

```python
def etags(resources, requests):
    tags = {p: hashlib.md5(c.encode()).hexdigest()[:8] for p, c in resources.items()}
    return [304 if tag == tags[p] else (200, tags[p]) for p, tag in requests]
```

**Cost:** O(n · L + r) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Hash the content on every request. That costs O(r · L) time and O(1) space."

"I can do better: Compute each ETag once. That's O(n · L + r) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/conditional-get-with-etags)

<a id="q17"></a>
### Q17. Retries Within a Deadline

🟡 Interview standard · *Timeouts must add up*

A request has an overall `deadline` (ms). Each try may take up to `try_timeout`; between tries the client waits the listed backoffs. Return how many tries can start and still finish before the deadline in the worst case.

**Examples**

- `deadline = 1000, try_timeout = 300, backoffs = [50, 100, 200, 400]` → `2`
- `deadline = 500, try_timeout = 600, backoffs = [10]` → `0`

In [ ]:
def tries_in_deadline(deadline, try_timeout, backoffs):
    """
    A request has an overall deadline (ms). Each try may take up to try_timeout;
    between tries the client waits the listed backoffs. Return how many tries
    can start and still finish before the deadline in the worst case.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(tries_in_deadline, [
    ((1000, 300, [50, 100, 200, 400]), 2),
    ((500, 600, [10]), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add time until the deadline is passed.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, with `itertools.accumulate`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Add time until the deadline is passed.

```python
def tries_in_deadline(deadline, try_timeout, backoffs):
    used, tries = 0, 0
    waits = [0] + list(backoffs)
    for w in waits:
        if used + w + try_timeout <= deadline:
            used += w + try_timeout
            tries += 1
        else:
            break
    return tries
```

**Cost:** O(b) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, with `itertools.accumulate`.

```python
def tries_in_deadline(deadline, try_timeout, backoffs):
    from itertools import accumulate, takewhile
    ends = accumulate(w + try_timeout for w in [0, *backoffs])
    return sum(1 for _ in takewhile(lambda e: e <= deadline, ends))
```

**Cost:** O(b) time, O(b) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add time until the deadline is passed. That costs O(b) time and O(1) space."

"I can do better: Same, with `itertools.accumulate`. That's O(b) time and O(b) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/retries-within-a-deadline)

<a id="q18"></a>
### Q18. An API Gateway

🟡 Interview standard · *Auth, then route*

For each request `(path, token)`: return 401 if the token is not in `valid`, else the service of the longest matching route prefix, else 404.

**Examples**

- `routes = {'/api/users': 'user-svc', '/api/users/admin': 'admin-svc', '/api/orders': 'order-svc'}, valid = {'t1', 't2'}, requests = [('/api/users/42', 't1'), ('/api/users/admin/x', 't2'), ('/api/orders', 'bad'), ('/api/payments', 't1')]` → `['user-svc', 'admin-svc', 401, 404]`

In [ ]:
def gateway(routes, valid, requests):
    """
    For each request (path, token): return 401 if the token is not in valid,
    else the service of the longest matching route prefix, else 404.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(gateway, [
    (({'/api/users': 'user-svc', '/api/users/admin': 'admin-svc', '/api/orders': 'order-svc'}, {'t1', 't2'}, [('/api/users/42', 't1'), ('/api/users/admin/x', 't2'), ('/api/orders', 'bad'), ('/api/payments', 't1')]), ['user-svc', 'admin-svc', 401, 404]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check every route for every request.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Try prefixes from longest to shortest.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Check every route for every request.

```python
def gateway(routes, valid, requests):
    out = []
    for path, token in requests:
        if token not in valid:
            out.append(401)
            continue
        best = None
        for prefix, svc in routes.items():
            if path.startswith(prefix) and (best is None or len(prefix) > len(best[0])):
                best = (prefix, svc)
        out.append(best[1] if best else 404)
    return out
```

**Cost:** O(r · k) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Try prefixes from longest to shortest.

```python
def gateway(routes, valid, requests):
    ordered = sorted(routes, key=len, reverse=True)
    def route(path):
        return next((routes[p] for p in ordered if path.startswith(p)), 404)
    return [401 if token not in valid else route(path) for path, token in requests]
```

**Cost:** O(r · k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check every route for every request. That costs O(r · k) time and O(1) space."

"I can do better: Try prefixes from longest to shortest. That's O(r · k) time and O(k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `routes = {'/a': 'A'}, valid = {'t'}, requests = [('/a/1', 't')]`

1. GeneratorExit raised: 
2. route() returns 'A'
3. gateway() returns ['A']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/an-api-gateway)

<a id="q19"></a>
### Q19. Sign Webhook Payloads (HMAC)

🟡 Interview standard · *Prove the sender*

Webhook receivers verify an HMAC-SHA256 signature of the body with a shared secret. Return the first 16 hex characters of the signature for each payload. The brute force builds HMAC by hand (inner and outer padded hashes); the optimal uses the `hmac` module.

**Examples**

- `secret = 's3cret', payloads = ['{"order":1,"status":"paid"}', '{"order":1,"status":"refunded"}']` → `['efb845265d0aec52', 'f86dbec2aaa2f23e']`

In [ ]:
import hmac

def sign_payloads(secret, payloads):
    """
    Webhook receivers verify an HMAC-SHA256 signature of the body with a shared
    secret. Return the first 16 hex characters of the signature for each
    payload. The brute force builds HMAC by hand (inner and outer padded
    hashes); the optimal uses the hmac module.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sign_payloads, [
    (('s3cret', ['{"order":1,"status":"paid"}', '{"order":1,"status":"refunded"}']), ['efb845265d0aec52', 'f86dbec2aaa2f23e']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

HMAC by hand: H((K ⊕ opad) ‖ H((K ⊕ ipad) ‖ m)).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`hmac.new(key, msg, hashlib.sha256)`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** HMAC by hand: H((K ⊕ opad) ‖ H((K ⊕ ipad) ‖ m)).

```python
def sign_payloads(secret, payloads):
    key = secret.encode().ljust(64, b"\0")
    ipad = bytes(b ^ 0x36 for b in key)
    opad = bytes(b ^ 0x5C for b in key)
    out = []
    for p in payloads:
        inner = hashlib.sha256(ipad + p.encode()).digest()
        out.append(hashlib.sha256(opad + inner).hexdigest()[:16])
    return out
```

**Cost:** O(L) time, O(L) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `hmac.new(key, msg, hashlib.sha256)`.

```python
def sign_payloads(secret, payloads):
    return [hmac.new(secret.encode(), p.encode(), hashlib.sha256).hexdigest()[:16] for p in payloads]
```

**Cost:** O(L) time, O(L) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: HMAC by hand: H((K ⊕ opad) ‖ H((K ⊕ ipad) ‖ m)). That costs O(L) time and O(L) space."

"I can do better: `hmac.new(key, msg, hashlib.sha256)`. That's O(L) time and O(L) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/sign-webhook-payloads-hmac)

<a id="q20"></a>
### Q20. A Circuit Breaker

🔴 Stretch · *Fail fast when a dependency is down*

Calls `(time, would_succeed)` go through a breaker. Closed: after `threshold` consecutive failures it opens. Open: calls are 'rejected' until `cooldown` seconds have passed since opening; then it is half-open and lets one call try: success closes it, failure reopens it. Return 'ok', 'fail' or 'rejected' per call.

**Examples**

- `calls = [(0, True), (1, False), (2, False), (3, False), (4, True), (5, True), (9, False), (10, True), (15, True), (16, True)], threshold = 3, cooldown = 5` → `['ok', 'fail', 'fail', 'fail', 'rejected', 'rejected', 'fail', 'rejected', 'ok', 'ok']`

In [ ]:
def circuit_breaker(calls, threshold, cooldown):
    """
    Calls (time, would_succeed) go through a breaker. Closed: after threshold
    consecutive failures it opens. Open: calls are 'rejected' until cooldown
    seconds have passed since opening; then it is half-open and lets one call
    try: success closes it, failure reopens it. Return 'ok', 'fail' or
    'rejected' per call.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(circuit_breaker, [
    (([(0, True), (1, False), (2, False), (3, False), (4, True), (5, True), (9, False), (10, True), (15, True), (16, True)], 3, 5), ['ok', 'fail', 'fail', 'fail', 'rejected', 'rejected', 'fail', 'rejected', 'ok', 'ok']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A state variable and if/elif branches.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same state machine, fewer branches.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A state variable and if/elif branches.

```python
def circuit_breaker(calls, threshold, cooldown):
    state, fails, opened, out = "closed", 0, None, []
    for t, ok in calls:
        if state == "open":
            if t - opened >= cooldown:
                state = "half"
            else:
                out.append("rejected")
                continue
        if ok:
            out.append("ok")
            fails = 0
            state = "closed"
        else:
            out.append("fail")
            fails += 1
            if state == "half" or fails >= threshold:
                state = "open"
                opened = t
    return out
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same state machine, fewer branches.

```python
def circuit_breaker(calls, threshold, cooldown):
    fails, opened, out = 0, None, []
    for t, ok in calls:
        if opened is not None and t - opened < cooldown:
            out.append("rejected")
            continue
        half = opened is not None
        out.append("ok" if ok else "fail")
        fails = 0 if ok else fails + 1
        opened = None if ok else (t if half or fails >= threshold else None)
    return out
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A state variable and if/elif branches. That costs O(n) time and O(1) space."

"I can do better: Same state machine, fewer branches. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/apis/a-circuit-breaker)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Design a rate limiter.</b></summary>

Clarify: per user or IP, limits, hard or soft. Put it in the gateway; use a token bucket or sliding window counter stored in Redis with atomic operations (Lua or INCR + EXPIRE); return 429 with Retry-After and rate-limit headers; fail open if Redis is down for non-critical limits.

</details>

<details>
<summary><b>2. Why cursor pagination?</b></summary>

It uses the index to jump straight to the next page (constant cost), and inserts or deletes don't shift pages, so nothing is skipped or repeated.

</details>

<details>
<summary><b>3. How do you make POST safe to retry?</b></summary>

Require an Idempotency-Key header; store key → response atomically with the change; return the stored response for repeats and reject reuse with a different body.

</details>

<details>
<summary><b>4. What does a circuit breaker solve?</b></summary>

It stops a failing dependency from tying up threads and connections with doomed calls, returns errors or fallbacks fast, and periodically tests whether the dependency has recovered.

</details>

<details>
<summary><b>5. How do you version an API?</b></summary>

Put the major version in the path (/v1) or a header, make additive changes without a new version, and deprecate old versions with notice before returning 410.

</details>

<details>
<summary><b>6. Polling, long polling, SSE or WebSockets?</b></summary>

Polling is simplest but wasteful; long polling holds the request until data arrives; SSE is one-way server push over HTTP; WebSockets are two-way, for chat and games.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Designing an API
- [ ] I can explain: Pagination, idempotency and caching
- [ ] I can explain: Rate limiting
- [ ] I can explain: Resilience at the edge

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Conditional GET With ETags
- [ ] [Q17](#q17) Retries Within a Deadline
- [ ] [Q18](#q18) An API Gateway
- [ ] [Q19](#q19) Sign Webhook Payloads (HMAC)
- [ ] [Q20](#q20) A Circuit Breaker

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.